# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the `mlcroissant` library. We will load metadata, inspect available record sets and fields (by `@id`), extract records for analysis, and perform basic exploratory data analysis (EDA) and visualization.

### Dataset Source
The dataset source is defined by a Croissant schema:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields (columns), and their `@id` values. Referencing entities by `@id` is crucial for unambiguous data operations with Croissant datasets.

Let's enumerate record sets and inspect their schemas:

In [ ]:
# List all record sets by @id
all_record_sets = list(dataset.record_sets)
if not all_record_sets:
    print("No record sets detected in this Croissant. Trying to discover from schema...")
    # fallback in case .record_sets is empty due to schema structure
    all_record_sets = [rset['@id'] for rset in getattr(metadata, 'recordSet', [])]
print(f"Record sets discovered in the dataset:")
for i, record_set in enumerate(dataset.record_sets):
    print(f"  {i+1}. @id: {record_set['@id']}  |  name: {record_set.get('name', '')}")
    # List field/column IDs for each recordSet
    if 'field' in record_set:
        print("      Fields:")
        for field in record_set['field']:
            print(f"        - @id: {field['@id']}  |  name: {field.get('name', '')}")

## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames for exploration. All operations will be performed by referring to record set and field `@id`.

First, select a primary record set to load (by @id). Replace the example `@id` values with those from previous cell's output for project-specific use:

In [ ]:
# Example: let's auto-detect the first tabular record set and its fields
recordsets = list(dataset.record_sets)
main_record_set_id = None
if recordsets:
    # Choose the record set that has fields
    for rs in recordsets:
        if 'field' in rs:
            main_record_set_id = rs['@id']
            break

if main_record_set_id is None:
    raise ValueError("No suitable record set found in this dataset Croissant schema.")
print(f"Selected record set: {main_record_set_id}")

# Load data for each discovered record set with fields
dataframes = {}
selected_record_set_fields = None
for record_set in dataset.record_sets:
    rset_id = record_set['@id']
    fields = [f['@id'] for f in record_set.get('field', [])]
    if fields:
        records = list(dataset.records(record_set=rset_id))
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        if rset_id == main_record_set_id:
            selected_record_set_fields = fields

print(f"Columns in the selected DataFrame (@id):\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's conduct some basic EDA using only the record set and field `@id` references.

- Filter records based on a numeric field (e.g., diagnosis interval, age, etc.)
- Normalize a numeric variable
- Group by a categorical field (e.g., sex, anatomical site, or cancer type)

Please replace `numeric_field_id` and `group_field_id` with relevant field `@id` values for this dataset.

In [ ]:
# Example: Map ID to a likely numeric field (first numerical column detected)
df = dataframes[main_record_set_id]

# Try to find a numeric field by raw type inspection:
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Try alternative: try to force convert each column to numeric
    for col in df.columns:
        try:
            if pd.to_numeric(df[col], errors='coerce').notnull().sum() > 0:
                numeric_field_id = col
                df[col] = pd.to_numeric(df[col], errors='coerce')
                break
        except Exception:
            pass
if numeric_field_id is None:
    raise ValueError("No numeric field found in record set.")
print(f"Using numeric field: {numeric_field_id}")

threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 10
# Filter for values above threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to find a grouping field (likely categorical, exclude pure numeric fields)
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() < 10:
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. All plots are referenced by field `@id`.

Here we show the (normalized) value distribution and, if grouped, a bar plot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field (before/after normalization)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True, ax=ax[0])
ax[0].set_title(f"Distribution of {numeric_field_id}")

if f"{numeric_field_id}_normalized" in filtered_df.columns:
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=12, kde=True, ax=ax[1])
    ax[1].set_title(f"Distribution of {numeric_field_id} (normalized)")
else:
    ax[1].set_visible(False)
plt.show()

# If group_field_id is available, show bar plot
if group_field_id:
    plt.figure(figsize=(8,4))
    group_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
    group_means.plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean of {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded FAIR² dataset metadata and records using `mlcroissant`, referencing all entities by their `@id` fields for reliability.
- Explored available record sets and columns; extracted tabular data as DataFrames.
- Performed EDA including filtering, normalization, grouping, and basic visualizations using only field `@id`s for all references.

This approach enables reproducible and robust dataset exploration across schema updates, supporting transparent research workflows.

_Replace field and group `@id` values with those suited for the clinical question or modeling pipeline you're developing from this dataset._